In [1]:
import pandas as pd

# 전처리 데이터 불러오기
df = pd.read_csv("data/processed/electricity_clean.csv")

print(df.shape)
print(df.head())

(110040, 7)
     연도     시도  시군구 계약종별  월        사용량       날짜
0  2021  서울특별시  종로구  주택용  1   25471662  2021-01
1  2021  서울특별시  종로구  일반용  1  113082168  2021-01
2  2021  서울특별시  종로구  교육용  1   10072542  2021-01
3  2021  서울특별시  종로구  산업용  1    6243796  2021-01
4  2021  서울특별시  종로구  농사용  1       5390  2021-01


In [2]:
# 군위군 데이터 확인
gunwi = df[df["시군구"] == "군위군"]

print(gunwi.groupby(["연도", "시도"]).size())

연도    시도   
2021  경상북도     96
2022  경상북도     96
2023  경상북도     96
      대구광역시    96
2024  대구광역시    96
2025  대구광역시    96
dtype: int64


In [3]:
# 개성시 데이터 확인
gaeseong = df[df["시군구"] == "개성시"]

print(gaeseong.groupby("연도")["사용량"].agg(
    ["count", "sum", "min", "max"]
))

      count  sum  min  max
연도                        
2021     48    0    0    0
2022     48    0    0    0


In [4]:
# 전처리
# 원본 데이터 보존
df_model = df.copy()

# 1. 개성시 제외
df_model = df_model[
    df_model["시군구"] != "개성시"
].copy()

# 2. 군위군 지역명 통일
df_model.loc[
    df_model["시군구"] == "군위군",
    "시도"
] = "대구광역시"

print("전처리 후 데이터 크기:", df_model.shape)

print(
    df_model[df_model["시군구"] == "군위군"]
    .groupby(["연도", "시도"])
    .size()
)

전처리 후 데이터 크기: (109944, 7)
연도    시도   
2021  대구광역시     96
2022  대구광역시     96
2023  대구광역시    192
2024  대구광역시     96
2025  대구광역시     96
dtype: int64


In [5]:
# 지역명 통일 후 중복 확인
keys = ["시도", "시군구", "계약종별", "날짜"]

duplicates = df_model[
    df_model.duplicated(subset=keys, keep=False)
]

print("중복 행 개수:", len(duplicates))

print(
    duplicates.groupby(["연도", "시군구"]).size()
)

중복 행 개수: 192
연도    시군구
2023  군위군    192
dtype: int64


In [6]:
#2023년 중복 데이터 통합
# 지역과 날짜, 계약종별 기준으로 사용량 합산
keys = ["연도", "시도", "시군구", "계약종별", "월", "날짜"]

df_model = (
    df_model
    .groupby(keys, as_index=False)["사용량"]
    .sum()
)

print("통합 후 데이터 크기:", df_model.shape)

통합 후 데이터 크기: (109848, 7)
